In [126]:
import geopandas as gpd
import pandas as pd
from libpysal.weights import Queen
from esda.moran import Moran_Local
import numpy as np

In [127]:
# --- Parameters ---
shapefile_path = "../../data/Local_Authority_Districts_(May_2025)_Boundaries_UK_BFC_(V2)/Local_Authority_Districts_(May_2025)_Boundaries_UK_BFC_(V2).shp"  # Change to your shapefile path
id_column = "LAD25CD"       # Column where first letter is E/W/S/N
name_column = "LAD25NM"   # Column with local authority names

# --- Load shapefile ---
gdf = gpd.read_file(shapefile_path)


gdf = gdf[gdf['LAD25CD'] != 'E06000046'] # Exclude Isle of Wight as no neighbours
gdf = gdf[gdf['LAD25CD'] != 'E06000053'] # Exclude Isles of Scilly as no neighbours

# Filter to only England
gdf = gdf[gdf[id_column].str[0].isin(["E"])]

region = pd.read_csv('../../data/Region lookup.csv')

gdf = gdf.merge(
    region[['LAD24CD', 'RGN24NM']].rename(columns={'LAD24CD': 'LAD25CD', 'RGN24NM': 'Region'}),
    on='LAD25CD', 
    how='left'
)


Centroids and rook contiguity neighbours

In [128]:
# Ensure consistent projection
gdf = gdf.to_crs(epsg=27700)  # British National Grid

gdf['area_km2'] = gdf.geometry.area / 1e6

# Compute centroids (in projected CRS)
gdf["centroid_x"] = gdf.geometry.centroid.x
gdf["centroid_y"] = gdf.geometry.centroid.y

# Build neighbor list (rook contiguity, optimized with spatial index)
rows = []
for idx, area in gdf.iterrows():
    # Use spatial index for speed
    possible_matches_index = list(gdf.sindex.intersection(area.geometry.bounds))
    possible_matches = gdf.iloc[possible_matches_index]

    # Find actual touching neighbors
    touching = possible_matches[possible_matches.geometry.touches(area.geometry)]

    # Append one row per neighbor
    for _, neighbor in touching.iterrows():
        rows.append({
            id_column: area[id_column],
            name_column: area[name_column],
            "neighbour_name": neighbor[name_column],
            "neighbour_id": neighbor[id_column],
            "centroid_x": area["centroid_x"],
            "centroid_y": area["centroid_y"]
        })

# Create DataFrame in long format
neighbors_df = pd.DataFrame(rows)



Load in Average prices, reduce to just LAs in England and remove Islands

In [129]:
la_list = gdf[['LAD25CD']]

hpi_raw = pd.read_csv(filepath_or_buffer="../../data/UK-HPI-full-file-2025-05.csv")
hpi_raw = hpi_raw[['Date', 'RegionName', 'AreaCode', 'AveragePrice']]
hpi_raw = hpi_raw[hpi_raw['AreaCode'].isin(la_list['LAD25CD'])]
hpi_raw['Date'] = pd.to_datetime(hpi_raw['Date'], format='%d/%m/%Y')

hpi_raw = hpi_raw[hpi_raw['AreaCode'] != 'E06000046'] # Exclude Isle of Wight as no neighbours
hpi_raw = hpi_raw[hpi_raw['AreaCode'] != 'E06000053'] # Exclude Isles of Scilly as no neighbours

Calculate the average neighbours average value

In [130]:
la_neighbour_avg = []

for area in hpi_raw['AreaCode'].unique():
    area_neighbour_df = neighbors_df[neighbors_df['LAD25CD'] == area].copy()
    area_df = hpi_raw[hpi_raw['AreaCode'].isin(area_neighbour_df['neighbour_id'])].copy()

    neighbour_avg = area_df.groupby('Date').agg(
        AverageNeighbourPrice = ('AveragePrice', 'mean')
    ).reset_index()

    neighbour_avg['RegionName'] = area_neighbour_df['LAD25NM'].iloc[0]
    neighbour_avg['AreaCode'] = area_neighbour_df['LAD25CD'].iloc[0]

    # Store for concatenation
    la_neighbour_avg.append(neighbour_avg)

# Combine all area_code DataFrames into one
la_neighbour_avg = pd.concat(la_neighbour_avg, ignore_index=True)



In [131]:
# Sort properly
la_neighbour_avg = la_neighbour_avg.sort_values(['RegionName', 'Date'])

# Create lag-1 column
la_neighbour_avg['AvgNeighbourPrice_lag1'] = (
    la_neighbour_avg.groupby('RegionName')['AverageNeighbourPrice']
      .shift(1)
)


In [132]:
la_neighbour_avg.query('AreaCode == "E06000058"')

,Date,AverageNeighbourPrice,RegionName,AreaCode,AvgNeighbourPrice_lag1
8030,1995-01-01,65862.0,"Bournemouth, Christchurch and Poole",E06000058,NaN
8031,1995-02-01,65903.5,"Bournemouth, Christchurch and Poole",E06000058,65862.0
8032,1995-03-01,65998.5,"Bournemouth, Christchurch and Poole",E06000058,65903.5
8033,1995-04-01,65711.5,"Bournemouth, Christchurch and Poole",E06000058,65998.5
8034,1995-05-01,65298.0,"Bournemouth, Christchurch and Poole",E06000058,65711.5
...,...,...,...,...,...
8390,2025-01-01,358144.0,"Bournemouth, Christchurch and Poole",E06000058,362056.5
8391,2025-02-01,361201.5,"Bournemouth, Christchurch and Poole",E06000058,358144.0
8392,2025-03-01,362854.0,"Bournemouth, Christchurch and Poole",E06000058,361201.5
8393,2025-04-01,360871.0,"Bournemouth, Christchurch and Poole",E06000058,362854.0


In [133]:
hpi = pd.merge(
    hpi_raw,
    la_neighbour_avg,
    on=['Date', 'AreaCode'],
    how='left'
)

In [134]:
w_queen = Queen.from_dataframe(gdf)  
w_queen.transform = 'r'  # Row-standardize weights

C:\Users\slong\AppData\Local\Temp\ipykernel_6820\3669738718.py:1: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w_queen = Queen.from_dataframe(gdf)


In [135]:
print(w_queen.neighbors)  # dict: {LA_index: [neighbor_indices]}
print(w_queen.weights)    # dict: {LA_index: [weights]}

{0: [3, 43], 1: [59, 2, 3], 2: [1, 59], 3: [0, 1, 4, 59, 43], 4: [59, 3, 43], 5: [45, 6, 235, 236, 237], 6: [5, 230, 233, 234, 44, 45, 237], 7: [225, 226, 135, 137, 141, 142, 143], 8: [136, 145], 9: [10], 10: [240, 9, 59, 12, 13], 11: [154, 12, 159], 12: [240, 168, 10, 11, 159], 13: [10, 59], 14: [73, 66, 70], 15: [152, 146, 147, 148], 16: [158, 148, 150, 55], 17: [169, 170, 173, 167], 18: [46, 101, 214], 19: [184, 46, 183], 20: [184, 185, 182], 21: [48, 22, 23, 24, 60], 22: [24, 21, 23], 23: [60, 21, 22], 24: [48, 100, 21, 22, 103], 25: [78], 26: [78, 79], 27: [48, 177, 100], 28: [64, 55, 157, 158, 63], 29: [50, 119], 30: [96, 90], 31: [89, 276, 87], 32: [130, 132, 126, 127], 33: [197, 110, 37, 38], 34: [176, 177, 114, 35, 48, 38, 105], 35: [176, 34, 38], 36: [37, 196, 277, 54], 37: [33, 195, 196, 197, 38, 36, 54], 38: [33, 34, 35, 37, 105, 110, 176, 54], 39: [49, 50, 54, 55, 56], 40: [210, 211, 84, 206], 41: [115, 108, 111], 42: [114, 107], 43: [0, 258, 3, 4, 51, 245, 58, 59], 44: [2

In [136]:
results = []

for month in hpi_raw['Date'].unique():
    month_data = hpi_raw[hpi_raw['Date'] == month]
    y = month_data['AveragePrice'].values
    
    moran_loc = Moran_Local(y, w_queen)
    
    month_results = pd.DataFrame({
        'local_I': moran_loc.Is,
        'p_value': moran_loc.p_sim,
        'quadrant': moran_loc.q
    }, index=month_data['AreaCode'].values)
    
    month_results['Date'] = month
    results.append(month_results)

local_moran_df = pd.concat(results)
local_moran_df.reset_index(inplace=True)

In [137]:
# Sort properly
local_moran_df = local_moran_df.sort_values(['index', 'Date'])

# Create lag-1 column
local_moran_df['local_I_lag1'] = (
    local_moran_df.groupby('index')['local_I']
      .shift(1)
)
local_moran_df['p_value_lag1'] = (
    local_moran_df.groupby('index')['p_value']
      .shift(1)
)
local_moran_df['quadrant_lag1'] = (
    local_moran_df.groupby('index')['quadrant']
      .shift(1)
)


In [138]:
local_moran_df

hpi = pd.merge(
    hpi,
    local_moran_df,
    left_on=['Date', 'AreaCode'],
    right_on=['Date', 'index'],
    how='left'
).drop(columns=['index'])

In [139]:
hpi

,Date,RegionName_x,AreaCode,AveragePrice,AverageNeighbourPrice,RegionName_y,AvgNeighbourPrice_lag1,local_I,p_value,quadrant,local_I_lag1,p_value_lag1,quadrant_lag1
0,1995-01-01,Adur,E07000223,54669,58617.75,Adur,NaN,0.081166,0.246,3,NaN,NaN,NaN
1,1995-02-01,Adur,E07000223,55864,58719.75,Adur,58617.75,0.049412,0.236,3,0.081166,0.246,3.0
2,1995-03-01,Adur,E07000223,55880,59421.50,Adur,58719.75,0.048777,0.238,3,0.049412,0.236,3.0
3,1995-04-01,Adur,E07000223,55596,60068.00,Adur,59421.50,0.059690,0.231,3,0.048777,0.238,3.0
4,1995-05-01,Adur,E07000223,53483,60124.00,Adur,60068.00,0.119831,0.213,3,0.059690,0.231,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
107305,2025-01-01,York,E06000014,305903,243202.00,York,242616.00,-0.084556,0.139,2,-0.077448,0.134,2.0
107306,2025-02-01,York,E06000014,304052,244940.50,York,243202.00,-0.103360,0.115,2,-0.084556,0.139,2.0
107307,2025-03-01,York,E06000014,305832,248447.50,York,244940.50,-0.103930,0.119,2,-0.103360,0.115,2.0
107308,2025-04-01,York,E06000014,307638,246783.50,York,248447.50,-0.098423,0.129,2,-0.103930,0.119,2.0


In [140]:
CoL_centroid = gdf.query('LAD25CD == "E09000001"')[['centroid_x', 'centroid_y']]
gdf_attach = gdf[['LAD25CD', 'LAD25NM', 'area_km2', 'centroid_x', 'centroid_y', 'Region']]
gdf_attach.loc[:, 'CoL_centroid_x'] = CoL_centroid['centroid_x'].iloc[0]
gdf_attach.loc[:, 'CoL_centroid_y'] = CoL_centroid['centroid_y'].iloc[0]
gdf_attach['CoL_distance_km'] = np.hypot(
    gdf_attach['centroid_x'] - gdf_attach['CoL_centroid_x'],
    gdf_attach['centroid_y'] - gdf_attach['CoL_centroid_y']
) / 1000  # Convert to km   

C:\Users\slong\AppData\Local\Temp\ipykernel_6820\1814108756.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gdf_attach.loc[:, 'CoL_centroid_x'] = CoL_centroid['centroid_x'].iloc[0]
C:\Users\slong\AppData\Local\Temp\ipykernel_6820\1814108756.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gdf_attach.loc[:, 'CoL_centroid_y'] = CoL_centroid['centroid_y'].iloc[0]
C:\Users\slong\AppData\Local\Temp\ipykernel_6820\1814108756.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fr

In [141]:
hpi = pd.merge(
    hpi,
    gdf_attach[['LAD25CD', 'area_km2', 'centroid_x', 'centroid_y', 'CoL_distance_km', 'Region']],
    left_on='AreaCode',
    right_on='LAD25CD',
    how='left'
).drop(columns=['LAD25CD'])

In [142]:
hpi_07 = hpi.query('Date > "2007-02-01"')

In [143]:
def compute_time_safe_fixed_effect(df, la_col="AreaCode", target_col="AveragePrice", date_col="Date"):
    """
    Computes a time-safe fixed effect for each row using only past information.
    """
    df = df.sort_values(date_col).copy()
    
    # Create storage
    fe_values = []
    
    # Running global mean
    running_sum = 0
    running_count = 0
    
    # Running LA-level sums
    la_sums = {}
    la_counts = {}
    
    for idx, row in df.iterrows():
        la = row[la_col]
        
        # Global FE
        global_mean = running_sum / running_count if running_count > 0 else row[target_col]
        la_mean = (la_sums.get(la, 0) / la_counts.get(la, 0)) if la in la_sums else global_mean
        
        fe = la_mean - global_mean
        fe_values.append(fe)
        
        # Add THIS row's target value for future calculations
        price = row[target_col]
        running_sum += price
        running_count += 1
        
        la_sums[la] = la_sums.get(la, 0) + price
        la_counts[la] = la_counts.get(la, 0) + 1
    
    df["LA_FE"] = fe_values
    return df

In [144]:

hpi_encode = compute_time_safe_fixed_effect(hpi_07)

hpi_encode = pd.get_dummies(hpi_encode, columns=["Region"], prefix="Region", drop_first=True, dtype=int)

hpi_encode

,Date,RegionName_x,AreaCode,AveragePrice,AverageNeighbourPrice,RegionName_y,AvgNeighbourPrice_lag1,local_I,p_value,quadrant,...,CoL_distance_km,LA_FE,Region_East of England,Region_London,Region_North East,Region_North West,Region_South East,Region_South West,Region_West Midlands,Region_Yorkshire and The Humber
146,2007-03-01,Adur,E07000223,202182,222076.250000,Adur,220545.250000,-0.044236,0.221,4,...,75.909867,0.000000,0,0,0,0,1,0,0,0
83366,2007-03-01,St Albans,E07000240,335322,248187.142857,St Albans,248294.857143,-0.588687,0.194,4,...,33.448078,0.000000,1,0,0,0,0,0,0,0
83001,2007-03-01,Spelthorne,E07000213,239493,305757.714286,Spelthorne,302966.428571,0.041802,0.384,1,...,27.966480,0.000000,0,0,0,0,1,0,0,0
12191,2007-03-01,Broxbourne,E07000095,225959,260769.500000,Broxbourne,257550.750000,0.154540,0.177,1,...,23.592339,0.000000,1,0,0,0,0,0,0,0
82636,2007-03-01,Southwark,E09000028,319437,300105.250000,Southwark,296603.500000,0.152483,0.368,1,...,4.797720,0.000000,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71904,2025-05-01,Rossendale,E07000125,188034,170252.166667,Rossendale,170892.666667,0.353589,0.151,3,...,283.622551,-123382.429937,0,0,0,1,0,0,0,0
72269,2025-05-01,Rother,E07000064,349166,357660.600000,Rother,354150.800000,0.048968,0.090,1,...,76.487714,1059.082775,0,0,0,0,1,0,0,0
72634,2025-05-01,Rotherham,E08000018,193243,197394.166667,Rotherham,197409.833333,0.363607,0.133,3,...,225.748728,-122146.526520,0,0,0,0,0,0,0,1
66794,2025-05-01,Oldham,E08000004,204063,213603.666667,Oldham,213774.166667,0.362586,0.145,3,...,262.660235,-121934.571801,0,0,0,1,0,0,0,0


In [145]:
sdlt_conditions = [
    #SDLT rules 03/2007 to 08/2008
    (hpi_encode["Date"].between("2007-03-01", "2008-08-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 125000),
    (hpi_encode["Date"].between("2007-03-01", "2008-08-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(125000, 250000, inclusive = "right")),
    (hpi_encode["Date"].between("2007-03-01", "2008-08-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(250000, 500000, inclusive = "right")),
    (hpi_encode["Date"].between("2007-03-01", "2008-08-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 500000),
    
    #SDLT rules 08/2008 to 12/2009
    (hpi_encode["Date"].between("2008-09-01", "2009-12-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 175000),
    (hpi_encode["Date"].between("2008-09-01", "2009-12-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(175000, 250000, inclusive = "right")),
    (hpi_encode["Date"].between("2008-09-01", "2009-12-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(250000, 500000, inclusive = "right")),
    (hpi_encode["Date"].between("2008-09-01", "2009-12-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 500000),
    
    #SDLT rules 01/2010 to 03/2011
    (hpi_encode["Date"].between("2010-01-01", "2011-03-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 125000),
    (hpi_encode["Date"].between("2010-01-01", "2011-03-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(125000, 250000, inclusive = "right")),
    (hpi_encode["Date"].between("2010-01-01", "2011-03-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(250000, 500000, inclusive = "right")),
    (hpi_encode["Date"].between("2010-01-01", "2011-03-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 500000),
    
    #SDLT rules 04/2011 to 11/2014
    (hpi_encode["Date"].between("2011-04-01", "2014-11-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 125000),
    (hpi_encode["Date"].between("2011-04-01", "2014-11-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(125000, 250000, inclusive = "right")),
    (hpi_encode["Date"].between("2011-04-01", "2014-11-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(250000, 500000, inclusive = "right")),
    (hpi_encode["Date"].between("2011-04-01", "2014-11-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(500000, 1e6, inclusive = "right")),
    (hpi_encode["Date"].between("2011-04-01", "2012-03-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 1e6),
    (hpi_encode["Date"].between("2012-04-01", "2014-11-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(1e6, 2e6, inclusive = "right")),
    (hpi_encode["Date"].between("2012-04-01", "2014-11-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 2e6),
    
    #SDLT rules 12/2014 to 06/2020
    (hpi_encode["Date"].between("2014-12-01", "2020-06-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 125000),
    (hpi_encode["Date"].between("2014-12-01", "2020-06-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(125000, 250000, inclusive = "right")),
    (hpi_encode["Date"].between("2014-12-01", "2020-06-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(250000, 925000, inclusive = "right")),
    (hpi_encode["Date"].between("2014-12-01", "2020-06-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(925000, 1.5e6, inclusive = "right")),
    (hpi_encode["Date"].between("2014-12-01", "2020-06-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 1.5e6),
    
    #SDLT rules 07/2020 to 06/2021
    (hpi_encode["Date"].between("2020-07-01", "2021-06-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 5e5),
    (hpi_encode["Date"].between("2020-07-01", "2021-06-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(5e5, 9.25e5, inclusive = "right")),
    (hpi_encode["Date"].between("2020-07-01", "2021-06-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(9.25e5, 1.5e6, inclusive = "right")),
    (hpi_encode["Date"].between("2020-07-01", "2021-06-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 1.5e6),
    
    #SDLT rules 07/2021 to 09/2021
    (hpi_encode["Date"].between("2021-07-01", "2021-09-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 2.5e5),
    (hpi_encode["Date"].between("2021-07-01", "2021-09-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(2.5e5, 9.25e5, inclusive = "right")),
    (hpi_encode["Date"].between("2021-07-01", "2021-09-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(9.25e5, 1.5e6, inclusive = "right")),
    (hpi_encode["Date"].between("2021-07-01", "2021-09-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 1.5e6),
    
    #SDLT rules 10/2021 to 09/2022
    (hpi_encode["Date"].between("2021-10-01", "2022-09-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 1.25e5),
    (hpi_encode["Date"].between("2021-10-01", "2022-09-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(1.25e5, 2.5e5, inclusive = "right")),
    (hpi_encode["Date"].between("2021-10-01", "2022-09-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(2.5e5, 9.25e5, inclusive = "right")),
    (hpi_encode["Date"].between("2021-10-01", "2022-09-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(9.25e5, 1.5e6, inclusive = "right")),
    (hpi_encode["Date"].between("2021-10-01", "2022-09-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 1.5e6),
    
    #SDLT rules 10/2022 to 03/2025
    (hpi_encode["Date"].between("2022-10-01", "2025-03-01", inclusive="both")) & (hpi_encode["AveragePrice"] <= 2.5e5),
    (hpi_encode["Date"].between("2022-10-01", "2025-03-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(2.5e5, 9.25e5, inclusive = "right")),
    (hpi_encode["Date"].between("2022-10-01", "2025-03-01", inclusive="both")) & (hpi_encode["AveragePrice"].between(9.25e5, 1.5e6, inclusive = "right")),
    (hpi_encode["Date"].between("2022-10-01", "2025-03-01", inclusive="both")) & (hpi_encode["AveragePrice"] > 1.5e6)
]

sdlt_choices = [
    #SDLT rules 03/2007 to 08/2008
    0, 1, 3, 4,

    #SDLT rules 08/2008 to 12/2009
    0, 1, 3, 4,

    #SDLT rules 01/2010 to 03/2011
    0, 1, 3, 4,

    #SDLT rules 04/2011 to 11/2014
    0, 1, 3, 4, 5, 5, 7,
    
    #SDLT rules 12/2014 to 06/2020
    0, 2, 5, 10, 12,

    #SDLT rules 07/2020 to 06/2021
    0, 5, 10, 12,

    #SDLT rules 07/2021 to 09/2021
    0, 5, 10, 12,

    #SDLT rules 10/2021 to 09/2022
    0, 2, 5, 10, 12,

    #SDLT rules 10/2022 to 03/2025
    0, 5, 10, 12
]

hpi_encode['sdlt_perc_threshold'] = np.select(sdlt_conditions, sdlt_choices)

In [146]:
hpi_encode

,Date,RegionName_x,AreaCode,AveragePrice,AverageNeighbourPrice,RegionName_y,AvgNeighbourPrice_lag1,local_I,p_value,quadrant,...,LA_FE,Region_East of England,Region_London,Region_North East,Region_North West,Region_South East,Region_South West,Region_West Midlands,Region_Yorkshire and The Humber,sdlt_perc_threshold
146,2007-03-01,Adur,E07000223,202182,222076.250000,Adur,220545.250000,-0.044236,0.221,4,...,0.000000,0,0,0,0,1,0,0,0,1
83366,2007-03-01,St Albans,E07000240,335322,248187.142857,St Albans,248294.857143,-0.588687,0.194,4,...,0.000000,1,0,0,0,0,0,0,0,3
83001,2007-03-01,Spelthorne,E07000213,239493,305757.714286,Spelthorne,302966.428571,0.041802,0.384,1,...,0.000000,0,0,0,0,1,0,0,0,1
12191,2007-03-01,Broxbourne,E07000095,225959,260769.500000,Broxbourne,257550.750000,0.154540,0.177,1,...,0.000000,1,0,0,0,0,0,0,0,1
82636,2007-03-01,Southwark,E09000028,319437,300105.250000,Southwark,296603.500000,0.152483,0.368,1,...,0.000000,0,1,0,0,0,0,0,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71904,2025-05-01,Rossendale,E07000125,188034,170252.166667,Rossendale,170892.666667,0.353589,0.151,3,...,-123382.429937,0,0,0,1,0,0,0,0,0
72269,2025-05-01,Rother,E07000064,349166,357660.600000,Rother,354150.800000,0.048968,0.090,1,...,1059.082775,0,0,0,0,1,0,0,0,0
72634,2025-05-01,Rotherham,E08000018,193243,197394.166667,Rotherham,197409.833333,0.363607,0.133,3,...,-122146.526520,0,0,0,0,0,0,0,1,0
66794,2025-05-01,Oldham,E08000004,204063,213603.666667,Oldham,213774.166667,0.362586,0.145,3,...,-121934.571801,0,0,0,1,0,0,0,0,0


In [147]:
hpi_encode = pd.get_dummies(hpi_encode, columns=["quadrant_lag1"], prefix="LMIQuadrantlag1", drop_first=True, dtype=int)

In [148]:
hpi_encode

,Date,RegionName_x,AreaCode,AveragePrice,AverageNeighbourPrice,RegionName_y,AvgNeighbourPrice_lag1,local_I,p_value,quadrant,...,Region_North East,Region_North West,Region_South East,Region_South West,Region_West Midlands,Region_Yorkshire and The Humber,sdlt_perc_threshold,LMIQuadrantlag1_2.0,LMIQuadrantlag1_3.0,LMIQuadrantlag1_4.0
146,2007-03-01,Adur,E07000223,202182,222076.250000,Adur,220545.250000,-0.044236,0.221,4,...,0,0,1,0,0,0,1,0,0,1
83366,2007-03-01,St Albans,E07000240,335322,248187.142857,St Albans,248294.857143,-0.588687,0.194,4,...,0,0,0,0,0,0,3,0,0,1
83001,2007-03-01,Spelthorne,E07000213,239493,305757.714286,Spelthorne,302966.428571,0.041802,0.384,1,...,0,0,1,0,0,0,1,0,0,0
12191,2007-03-01,Broxbourne,E07000095,225959,260769.500000,Broxbourne,257550.750000,0.154540,0.177,1,...,0,0,0,0,0,0,1,0,0,0
82636,2007-03-01,Southwark,E09000028,319437,300105.250000,Southwark,296603.500000,0.152483,0.368,1,...,0,0,0,0,0,0,3,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71904,2025-05-01,Rossendale,E07000125,188034,170252.166667,Rossendale,170892.666667,0.353589,0.151,3,...,0,1,0,0,0,0,0,0,1,0
72269,2025-05-01,Rother,E07000064,349166,357660.600000,Rother,354150.800000,0.048968,0.090,1,...,0,0,1,0,0,0,0,0,0,0
72634,2025-05-01,Rotherham,E08000018,193243,197394.166667,Rotherham,197409.833333,0.363607,0.133,3,...,0,0,0,0,0,1,0,0,1,0
66794,2025-05-01,Oldham,E08000004,204063,213603.666667,Oldham,213774.166667,0.362586,0.145,3,...,0,1,0,0,0,0,0,0,1,0


In [149]:
hpi_data = hpi_encode

In [150]:
hpi_data

,Date,RegionName_x,AreaCode,AveragePrice,AverageNeighbourPrice,RegionName_y,AvgNeighbourPrice_lag1,local_I,p_value,quadrant,...,Region_North East,Region_North West,Region_South East,Region_South West,Region_West Midlands,Region_Yorkshire and The Humber,sdlt_perc_threshold,LMIQuadrantlag1_2.0,LMIQuadrantlag1_3.0,LMIQuadrantlag1_4.0
146,2007-03-01,Adur,E07000223,202182,222076.250000,Adur,220545.250000,-0.044236,0.221,4,...,0,0,1,0,0,0,1,0,0,1
83366,2007-03-01,St Albans,E07000240,335322,248187.142857,St Albans,248294.857143,-0.588687,0.194,4,...,0,0,0,0,0,0,3,0,0,1
83001,2007-03-01,Spelthorne,E07000213,239493,305757.714286,Spelthorne,302966.428571,0.041802,0.384,1,...,0,0,1,0,0,0,1,0,0,0
12191,2007-03-01,Broxbourne,E07000095,225959,260769.500000,Broxbourne,257550.750000,0.154540,0.177,1,...,0,0,0,0,0,0,1,0,0,0
82636,2007-03-01,Southwark,E09000028,319437,300105.250000,Southwark,296603.500000,0.152483,0.368,1,...,0,0,0,0,0,0,3,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71904,2025-05-01,Rossendale,E07000125,188034,170252.166667,Rossendale,170892.666667,0.353589,0.151,3,...,0,1,0,0,0,0,0,0,1,0
72269,2025-05-01,Rother,E07000064,349166,357660.600000,Rother,354150.800000,0.048968,0.090,1,...,0,0,1,0,0,0,0,0,0,0
72634,2025-05-01,Rotherham,E08000018,193243,197394.166667,Rotherham,197409.833333,0.363607,0.133,3,...,0,0,0,0,0,1,0,0,1,0
66794,2025-05-01,Oldham,E08000004,204063,213603.666667,Oldham,213774.166667,0.362586,0.145,3,...,0,1,0,0,0,0,0,0,1,0


In [151]:
hpi_data.to_excel("../../data/processed_hpi_with_bordering_las.xlsx", index=False)